<a href="https://colab.research.google.com/github/Youssef080808/AI_Agents/blob/main/MultiAgentSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
%%capture
!pip install 'smolagents[litellm]' plotly geopandas shapely kaleido -q
!pip install smolagents -U
!pip install ddgs

## Model

In [9]:
import os
from PIL import Image
from smolagents import CodeAgent, TransformersModel, GoogleSearchTool, VisitWebpageTool

model = TransformersModel(
    model_id="Qwen/Qwen2.5-Coder-3B-Instruct",
    device_map="auto",
    max_new_tokens=2048,
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [10]:
import math
from typing import Optional, Tuple

from smolagents import tool


@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))

22.82


## Agents

In [11]:
task = """Find 5 Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe with final_answer.
Also give me some supercar factories with the same cargo plane transfer time. Rules: Use tools given to you only; Never try to process strings using code: when you have a string to read, just print it and you'll see it."""

### Simple Agent
calculates cargo trave time

In [12]:
from google.colab import userdata
import os
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY').strip()

agent = CodeAgent(
    model=model,
    tools=[GoogleSearchTool("serper"), VisitWebpageTool(max_output_length=4000), calculate_cargo_travel_time],
    additional_authorized_imports=["pandas"],
    max_steps=6,
    max_print_outputs_length=3000,
)

In [ ]:
result = agent.run(task)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find 5 Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in  │
│ Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe with final_answer.                 │
│ Also give me some supercar factories with the same cargo plane transfer time. Rules: Use tools given to you     │
│ only                                                                                                            │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-3B-Instruct ────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### 1. Facts Survey

#### 1.1. Facts given in the task
- We need to find 5 Batman filming locations in the world.
- We need to calculate the time to transfer via cargo plane to each of these locations from Gotham, located at 
40.7128° N, 74.0060° W.
- We need to return the results in a pandas dataframe.

#### 1.2. Facts to look up
- **Batman Filming Locations**: To find the locations, we need to search for known Batman film locations around the
world.
  - **Source**: Wikipedia articles on Batman films and their production locations.
  - **Search Query**: "Batman film locations"

- **Cargo Plane Transfer Time**: To calculate the time, we need to use the great-circle distance formula and the 
given cruising speed.
  - **Source**: Aeronautical databases or online calculators.
  - **Search Query**: "Cargo plane transfer time calculation formula"

#### 1.3. Facts to derive
- **Final Answer**: A pandas dataframe containing the Batman filming locations, their respective distances from 
Gotham, and the calculated transfer times.

### 2. Plan

1. **Web Search for Batman Filming Locations**
   - Use `web_search` to find and extract information about known Batman film locations.
   - Extract the latitude and longitude for each location.
   - Store the results in a list of tuples.

2. **Visit Webpage for Cargo Plane Transfer Time Information**
   - Use `web_search` to find and extract information about cargo plane transfer times.
   - Identify the cruising speed used in the calculations.
   - Store the results in a dictionary.

3. **Calculate Transfer Times**
   - For each Batman filming location, use the `calculate_cargo_travel_time` function to compute the travel time.
   - Use the extracted latitude and longitude from step 1 and the cruising speed from step 2.

4. **Store Results in a Pandas DataFrame**
   - Create a pandas dataframe with columns for the location name, latitude, longitude, and transfer time.
   - Populate the dataframe with the data obtained from steps 1, 3, and 4.

5. **Final Answer**
   - Use `final_answer` to provide the final answer in the form of a pandas dataframe.

```python
# Step 1: Web Search for Batman Filming Locations
batman_locations = web_search("Batman film locations", None)

# Step 2: Visit Webpage for Cargo Plane Transfer Time Information
cargo_plane_info = web_search("Cargo plane transfer time calculation formula", None)

# Step 3: Calculate Transfer Times
cruising_speed_kmh = 750  # Typical cruising speed for cargo planes
transfer_times = []
for location in batman_locations:
    coords = (location['latitude'], location['longitude'])
    transfer_time = calculate_cargo_travel_time((40.7128, 74.0060), coords, cruising_speed_kmh)
    transfer_times.append(transfer_time)

# Step 4: Store Results in a Pandas DataFrame
import pandas as pd

df = pd.DataFrame({
    'Location': [location['name'] for location in batman_locations],
    'Latitude': [location['latitude'] for location in batman_locations],
    'Longitude': [location['longitude'] for location in batman_locations],
    'Transfer Time (hours)': transfer_times
})

# Step 5: Final Answer
final_answer(df)
```

'
```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import pandas as pd                                                                                              
                                                                                                                   
  # Step 1: Web Search for Batman Filming Locations                                                                
  batman_locations = web_search("Batman film locations", None)                                                     
                                                                                                                   
  # Step 2: Visit Webpage for Cargo Plane Transfer Time Information                                                
  cargo_plane_info = web_search("Cargo plane transfer time calculation formula", None)                             
                                                                                                                   
  # Step 3: Calculate Transfer Times                                                                               
  cruising_speed_kmh = 750  # Typical cruising speed for cargo planes                                              
  transfer_times = []                                                                                              
  for location in batman_locations:                                                                                
      coords = (location['latitude'], location['longitude'])                                                       
      transfer_time = calculate_cargo_travel_time((40.7128, 74.0060), coords, cruising_speed_kmh)                  
      transfer_times.append(transfer_time)                                                                         
                                                                                                                   
  # Step 4: Store Results in a Pandas DataFrame                                                                    
  df = pd.DataFrame({                                                                                              
      'Location': [location['name'] for location in batman_locations],                                             
      'Latitude': [location['latitude'] for location in batman_locations],                                         
      'Longitude': [location['longitude'] for location in batman_locations],                                       
      'Transfer Time (hours)': transfer_times                                                                      
  })                                                                                                               
                                                                                                                   
  # Step 5: Final Answer                                                                                           
  final_answer(df)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'for location in batman_locations:
    coords = (location['latitude'], location['longitude'])
    transfer_time = calculate_cargo_travel_time((40.7128, 74.0060), coords, cruising_speed_kmh)
    transfer_times.append(transfer_time)' due to: InterpreterError: Could not index # with 'latitude': TypeError: 
string indices must be integers, not 'str'

[Step 1: Duration 600.42 seconds| Input tokens: 3,108 | Output tokens: 556]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import pandas as pd                                                                                              
                                                                                                                   
  # Step 1: Web Search for Batman Filming Locations                                                                
  batman_locations = web_search("Batman film locations", None)                                                     
                                                                                                                   
  # Step 2: Visit Webpage for Cargo Plane Transfer Time Information                                                
  cargo_plane_info = web_search("Cargo plane transfer time calculation formula", None)                             
                                                                                                                   
  # Step 3: Calculate Transfer Times                                                                               
  cruising_speed_kmh = 750  # Typical cruising speed for cargo planes                                              
  transfer_times = []                                                                                              
  for location in batman_locations:                                                                                
      coords = (float(location['latitude']), float(location['longitude']))                                         
      transfer_time = calculate_cargo_travel_time((40.7128, 74.0060), coords, cruising_speed_kmh)                  
      transfer_times.append(transfer_time)                                                                         
                                                                                                                   
  # Step 4: Store Results in a Pandas DataFrame                                                                    
  df = pd.DataFrame({                                                                                              
      'Location': [location['name'] for location in batman_locations],                                             
      'Latitude': [location['latitude'] for location in batman_locations],                                         
      'Longitude': [location['longitude'] for location in batman_locations],                                       
      'Transfer Time (hours)': transfer_times                                                                      
  })                                                                                                               
                                                                                                                   
  # Step 5: Final Answer                                                                                           
  final_answer(df)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'for location in batman_locations:
    coords = (float(location['latitude']), float(location['longitude']))
    transfer_time = calculate_cargo_travel_time((40.7128, 74.0060), coords, cruising_speed_kmh)
    transfer_times.append(transfer_time)' due to: InterpreterError: Could not index # with 'latitude': TypeError: 
string indices must be integers, not 'str'

[Step 2: Duration 676.00 seconds| Input tokens: 6,966 | Output tokens: 1,161]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

#### Plan
give agent planning capabilities

In [14]:
agent.planning_interval = 4

detailed_report = agent.run(f"""
You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.

{task}
""")

print(detailed_report)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You're an expert analyst. You make comprehensive reports after visiting many websites.                          │
│ Don't hesitate to search for many queries at once in a for loop.                                                │
│ For each data point that you find, visit the source url to confirm numbers.                                     │
│                                                                                                                 │
│ Find 5 Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in  │
│ Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe with final_answer.                 │
│ Also give me some supercar factories with the same cargo plane transfer time. Rules: Use tools given to you     │
│ only                                                                                                            │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-3B-Instruct ────────────────────────────────────────────────────────────╯

KeyboardInterrupt: 

## Multi-Agent System

### Web Agent

In [ ]:
web_agent = CodeAgent(
    model=model,
    tools=[
        GoogleSearchTool(provider="serper"),
        VisitWebpageTool(max_ouptut_lengt=4000),
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=0,
    max_steps=5,
    max_print_outputs_length=3000,
)

### Manager Agent

#### Model

In [ ]:
!pip install bitsandbytes

In [ ]:

reasoning_model = TransformersModel(
    model_id="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    device_map="auto",
    max_new_tokens=2048,
)

In [ ]:
from smolagents.utils import encode_image_base64, make_image_url
from smolagents import OpenAIServerModel

def check_plot_saved(final_answer, agent_memory):
    assert os.path.exists("saved_map.png"), "Make sure to save the plot under saved_map.png!"
    return True

manager_agent = CodeAgent(
    model=reasoning_model,
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=[
        "geopandas",
        "plotly",
        "shapely",
        "json",
        "pandas",
        "numpy",
    ],
    planning_interval=5,
    verbosity_level=2,
    final_answer_checks=[check_plot_saved],
    max_steps=15,
)

In [ ]:
manager_agent.visualize()

In [ ]:
manager_agent.run("""
Find 5 Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.png!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
fig.show()
fig.write_image("saved_map.png")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
""")

In [ ]:
manager_agent.python_executor.state["fig"]